<img align="right" src="images/tf.png" width="128"/>
<img align="right" src="images/logo.png" width="128"/>

# Tutorial

This notebook gets you started with using
[Text-Fabric](https://annotation.github.io/text-fabric/) for coding in the Vulgate.

Familiarity with the underlying
[data model](https://annotation.github.io/text-fabric/tf/about/datamodel.html)
is recommended.

## Installing Text-Fabric

See [here](https://annotation.github.io/text-fabric/tf/about/install.html)

## Tip
If you start computing with this tutorial, first copy its parent directory to somewhere else,
outside your repository.
If you pull changes from the repository later, your work will not be overwritten.
Where you put your tutorial directory is up to you.
It will work from any directory.

## Data

This tutorial works with a clone of the repository in `~/github/rdtaylorjr/vulgate`.
Once the data is published on GitHub, Text-Fabric will fetch the data set for you
with `use("rdtaylorjr/vulgate")`, and store it in the `text-fabric-data` in your home directory.

# Features
The data of the corpus is organized in features.
They are *columns* of data.
Think of the corpus as a gigantic spreadsheet, where row 1 corresponds to the
first word, row 2 to the second word, and so on, for all words,
followed by the nodes of other types.

The information which text each word has, constitutes a column in that spreadsheet.

Instead of putting that information in one big table, the data is organized in separate columns.
We call those columns **features**.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import collections

# Incantation

The simplest way to get going is by this *incantation*:

In [3]:
from tf.app import use

In [4]:
A = use("rdtaylorjr/vulgate:clone", checkout="clone", hoist=globals())

**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,39,10233.92,100
chapter,915,436.20,100
verse,22805,17.50,100
word,399123,1.00,100


You can see which features have been loaded, and if you click on a feature name, you find its documentation.
If you hover over a name, you see where the feature is located on your system.

## API

The result of the incantation is that we have a bunch of special variables at our disposal
that give us access to the text and data of the corpus.

At this point it is helpful to throw a quick glance at the text-fabric API documentation
(see the links under **API Members** above).

The most essential thing for now is that we can use `F` to access the data in the features
we've loaded.
But there is more, such as `N`, which helps us to walk over the text, as we see in a minute.

The **API members** above show you exactly which new names have been inserted in your namespace.
If you click on these names, you go to the API documentation for them.

# Counting

In order to get acquainted with the data, we start with the simple task of counting.

## Count all nodes
We use the
[`N.walk()` generator](https://annotation.github.io/text-fabric/tf/core/nodes.html#tf.core.nodes.Nodes.walk)
to walk through the nodes.

In [5]:
A.indent(reset=True)
A.info("Counting nodes ...")

i = 0
for n in N.walk():
    i += 1

A.info("{} nodes".format(i))

  0.00s Counting nodes ...


  0.02s 422882 nodes


## What are those nodes?
Every node has a type, like word, verse, chapter, book.
But what exactly are they?

Text-Fabric has two special features, `otype` and `oslots`, that must occur in every Text-Fabric data set.
`otype` tells you for each node its type, and you can ask for the number of `slot`s in the text.

Here we go!

In [6]:
F.otype.slotType

'word'

In [7]:
F.otype.maxSlot

399123

In [8]:
F.otype.maxNode

422882

In [9]:
F.otype.all

('book', 'chapter', 'verse', 'word')

In [10]:
C.levels.data

(('book', 10233.923076923076, 399124, 399162),
 ('chapter', 436.2, 399163, 400077),
 ('verse', 17.501556676167507, 400078, 422882),
 ('word', 1, 1, 399123))

This is interesting: above you see all the textual objects, with the average size of their objects,
the node where they start, and the node where they end.

## Count individual object types
This is an intuitive way to count the number of nodes in each type.
Note in passing, how we use the `indent` in conjunction with `info` to produce neat timed
and indented progress messages.

In [11]:
A.indent(reset=True)
A.info("counting objects ...")

for otype in F.otype.all:
    i = 0

    A.indent(level=1, reset=True)

    for n in F.otype.s(otype):
        i += 1

    A.info("{:>7} {}s".format(i, otype))

A.indent(level=0)
A.info("Done")

  0.00s counting objects ...


   |     0.00s      39 books


   |     0.00s     915 chapters


   |     0.00s   22805 verses


   |     0.06s  399123 words


  0.06s Done


# Sections

The sections are *book*, *chapter*, and *verse*.
The Psalms follow the Greek numbering of the Vulgate: Psalm 22 is Psalm 23 of the Hebrew Bible.

In [12]:
v = T.nodeFromSection(("Psalms", 22, 1))
T.sectionFromNode(v)

('Psalms', 22, 1)

In [13]:
A.plain(v)

In [14]:
T.text(v)

'canticum David Dominus pascit me nihil mihi deerit '

Book names are in English by default.
The Latin names are in the feature `book`.

In [15]:
[(F.book.v(b), Fs("book@en").v(b)) for b in F.otype.s("book")][:5]

[('Genesis', 'Genesis'),
 ('Exodus', 'Exodus'),
 ('Leviticus', 'Leviticus'),
 ('Numeri', 'Numbers'),
 ('Deuteronomium', 'Deuteronomy')]

# Feature statistics

`F`
gives access to all features.
Every feature has a method
`freqList()`
to generate a frequency list of its values, higher frequencies first.

In [16]:
F.word.freqList()[:20]

(('et', 33053),
 ('in', 14415),
 ('ad', 5724),
 ('est', 5232),
 ('non', 4364),
 ('eius', 3845),
 ('qui', 3767),
 ('de', 3539),
 ('Dominus', 3162),
 ('ut', 2886),
 ('super', 2515),
 ('cum', 2358),
 ('autem', 2348),
 ('Israhel', 2275),
 ('me', 2050),
 ('sunt', 1974),
 ('quae', 1965),
 ('quod', 1845),
 ('eum', 1764),
 ('Domini', 1676))

# Sources

The words of most verses come from LatinISE, and the words of the chapters that LatinISE lacks
come from the IntraText Digital Library.
The feature `source` of a verse tells which.

In [17]:
F.source.freqList()

(('latinise', 21596), ('intratext', 1209))

In [18]:
for b in F.otype.s("book"):
    sources = collections.Counter(F.source.v(v) for v in L.d(b, otype="verse"))
    if sources["intratext"]:
        print(Fs("book@en").v(b), dict(sources))

Numbers {'intratext': 1156}
Zephaniah {'intratext': 53}


# Search

Text-Fabric has a search engine.
Here we look for verses with the word `Dominus`.

In [19]:
query = """
verse
  word word=Dominus
"""
results = A.search(query)

  0.19s 3162 results


In [20]:
A.table(results, end=5)

n,p,verse,word
1,Genesis 2:4,,Dominus
2,Genesis 2:5,,Dominus
3,Genesis 2:7,,Dominus
4,Genesis 2:8,,Dominus
5,Genesis 2:9,,Dominus


# All steps

* **start** your first step in mastering the Vulgate computationally
* **[search](search.ipynb)** turbo charge your hand-coding with search templates